### Import Libraries

In [1]:
# Import necessary libraries and modules
import os
import glob
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.prompts import MessagesPlaceholder

In [2]:
# Load api key from .env file
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

print("API key loaded")

API key loaded


#### Documents collections

In [3]:
# Load PDF documents from a specified folder
documents = []

for pdf_path in glob.glob("documents/*.pdf"):  # adjust folder path
    loader = PyPDFLoader(pdf_path)
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(documents)} PDF Documents.")

Loaded 6 PDF Documents.


#### Text Splitters

In [4]:
# Create splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=100,
    separators=["\n\n", "\n", ".", " ", ""]
)

# Split documents
chunks = text_splitter.split_documents(documents)

print(f"Split {len(documents)} documents into {len(chunks)} chunks")
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}: {chunk.page_content}")

Split 6 documents into 18 chunks

Chunk 1: Projects: 
Project 1: Car Price Prediction System 
• Developed a supervised machine learning model to predict car prices using 
historical sales data. 
• Performed data cleaning, feature engineering, model training, and evaluation 
to ensure high predictive accuracy. 
• Deployed the model via Flask for real-time price estimation. 
Technologies: Python, Pandas, Scikit-learn, Flask 
Project 2: Retail Sales Forecasting 
• Built a time-series forecasting system to predict monthly sales trends for an e-
commerce platform. 
• Implemented ARIMA and machine learning-based models to capture seasonal

Chunk 2: commerce platform. 
• Implemented ARIMA and machine learning-based models to capture seasonal 
patterns and improve prediction accuracy. 
Technologies: Python, Pandas, Statsmodels, Scikit-learn 
Project 3: Retrieval-Augmented Generation (RAG) Chatbot 
• Designed a document-based question answering system leveraging 
LangChain, vector embeddings, a

### Embeddings

In [5]:
# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=api_key
)

# Test embedding
test_embedding = embeddings.embed_query("What is RAG?")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

Embedding dimension: 1536
First 5 values: [0.0006281227106228471, 0.02569717727601528, 0.007161187008023262, 0.03336399793624878, -0.031968604773283005]


### Vector Store

In [6]:
# Create vector store from documents
vectorstore = Chroma.from_documents(
    chunks,
    embeddings,
    collection_name="my_info_collection",
    persist_directory="./chroma_db"
)

In [25]:
# Test retriver
query = "What projects has Olasunkanmi worked on?"

retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

results = retriever.invoke(query)
results

[Document(metadata={'producer': 'Microsoft® Word LTSC', 'page_label': '1', 'source': 'documents\\Personal Biography.pdf', 'creationdate': '2025-12-14T15:37:43+01:00', 'title': 'Personal Biography', 'total_pages': 1, 'creator': 'Microsoft® Word LTSC', 'page': 0, 'author': 'olasunkanmi', 'moddate': '2025-12-14T15:37:43+01:00'}, page_content='I, Olasunkanmi Akeem Rasak, am an AI Engineer with a strong foundation in mathematics, data \nscience, and applied machine learning, specializing in building intelligent systems that combine \npredictive modeling, natural language processing, and data-driven decision-making. My expertise spans \nreal-world AI applications in finance, e-commerce, and education, with hands-on experience designing, \ntraining, and deploying scalable machine learning models. \nI hold a Bachelor’s degree in Pure and Applied Mathematics from Ahmadu Bello University, Zaria, and'),
 Document(metadata={'page_label': '1', 'creationdate': '2025-12-14T15:38:25+01:00', 'title': '

### Conversational RAG

In [18]:
# Create LLM
llm = ChatOpenAI(
    model="gpt-3.5-turbo",
    temperature=0,
    openai_api_key=api_key
)

# Create retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# Prompt
prompt = ChatPromptTemplate.from_template("""
You are an AI assistant answering questions about Olasunkanmi Akeem Rasak using the provided documents.

Use ONLY the context below to answer the question.
If the answer is not in the context, say "I don't know."

<context>
{context}
</context>

Question: {question}

Answer in clear sentences.
At the end, list the sources you used as bullet points.
""")

# format documents
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')}\n{doc.page_content}"
        for doc in docs
    )

# RAG chain Using LCEL
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [24]:
# Test RAG chain
query = "What forex projects has Olasunkanmi worked on?"
response = rag_chain.invoke(query)
print(response)

Olasunkanmi Akeem Rasak has worked on a Forex Trading Strategy Automation project where he built an automated system for generating and executing Forex trading strategies based on predictive models.

Sources:
- documents\AI & ML Projects.pdf


#### Conversational RAG

In [26]:
# Store for chat histories
chat_store = {}

def get_session_history(session_id: str):
    if session_id not in chat_store:
        chat_store[session_id] = InMemoryChatMessageHistory()
    return chat_store[session_id]

# Create conversational prompt
conv_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an AI assistant answering questions about Olasunkanmi Akeem Rasak using the provided documents. Use ONLY the context below to answer the question. If the answer is not in the context, say I don't know."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("system", "Answer in clear sentences. At the end, list the sources you used as bullet points."),
    ("human", "Context: {context}\n\nQuestion: {question}")
])

# Build base chain
conv_chain_base = (
    RunnableParallel(
        context=lambda x: format_docs(retriever.invoke(x["question"])),
        question=lambda x: x["question"],
        chat_history=lambda x: x.get("chat_history", [])
    )
    | conv_prompt
    | llm
    | StrOutputParser()
)

# Wrap with message history
conv_chain = RunnableWithMessageHistory(
    conv_chain_base,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)


### Questions!!!

In [28]:
# First question
response = conv_chain.invoke(
    {"question": "What AI projects has Olasunkanmi worked on?"},
    config={"configurable": {"session_id": "user_1"}}
)
print("Response 1:\n", response)

# Follow-up question
response2 = conv_chain.invoke(
    {"question": "Which of those involve RAG systems?"},
    config={"configurable": {"session_id": "user_1"}}
)

print("\nResponse 2:\n", response2)

Response 1:
 Olasunkanmi Akeem Rasak has worked on the following AI projects:
- Sales forecasting
- Car price prediction
- Student performance analytics
- Forex trading strategy automation
- Hybrid car scanner automation
- Document-based AI systems

Sources:
- documents\Personal Biography.pdf
- documents\Professional Resume.pdf

Response 2:
 Olasunkanmi Akeem Rasak has worked on the following projects involving Retrieval-Augmented Generation (RAG) systems:
- Retrieval-Augmented Generation (RAG) Systems
- Retrieval-Augmented Generation (RAG) Chatbot

Sources:
- documents\Professional Resume.pdf
- documents\AI & ML Projects.pdf
